# Re-ranker, second round

Against the first round (`ranker.ipynb`) the candidate lists grow to the top 60 of every collaborative source (top 20 for content) and the top 60 popular programmes. Three experiments, all trained on labels before week 21 and compared on S21: E1 trains on S16 alone, E2 on S14, S16 and S18, E3 is E2 with a more regularised parameter set. The best experiment is refitted for the test window on all snapshots whose labels end by week 30.

In [1]:
# Environment, data, snapshots, collaborative models and candidate tables
import gc
import json
import os
import platform
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))
from sweettv import baselines, cf, data, features, ranker, recs, snapshot  # noqa: E402

RAW, CACHE = os.path.join(ROOT, 'data', 'raw'), os.path.join(ROOT, 'data', 'cache')
SUB_DIR = os.path.join(ROOT, 'submissions')
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
print(platform.platform(), '| python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| lightgbm', lgb.__version__)
T0 = time.time()
views = data.load_views(RAW, CACHE)
sched = data.load_schedule(RAW, CACHE)
test_users = pd.Index(data.load_submission(RAW).user_id)
SETTINGS = json.load(open(os.path.join(ROOT, 'modelling', 'cf_results.json')))['settings']
print('collaborative settings chosen in cf_models:', SETTINGS)
lab = sched[sched.labelled]
prog = lab.groupby('tv_show_id').agg(title=('tv_show_title', 'first'), category=('tv_show_category', 'first'),
                                     genre_1=('tv_show_genre_1', 'first'), genre_2=('tv_show_genre_2', 'first'),
                                     genre_3=('tv_show_genre_3', 'first'), year=('year_of_production', 'first'),
                                     director=('director', 'first'), actors=('actors', 'first'))
SNAPS = {'S14': (14, 20), 'S16': (16, 20), 'S18': (18, 20), 'S21': (21, 30), 'S26': (26, 30), 'S31': (31, 42)}
N_CF = {'als': 60, 'ppr': 60, 'rules': 60, 'content': 20, 'habit': 60}
N_POP = 60
snaps = {name: snapshot.build(views, sched, cut, end, with_target=(name != 'S31')) for name, (cut, end) in SNAPS.items()}
fallbacks = {name: baselines.popularity_ranking(s) for name, s in snaps.items()}


def cf_models(name):
    m = cf.Interactions.from_snapshot(snaps[name])
    als_kw = dict(SETTINGS['als'])
    als_kw['factors'] = int(als_kw['factors'])
    return {'als': cf.als(m, **als_kw), 'ppr': cf.ppr(m, **SETTINGS['ppr']), 'rules': cf.association_rules(m, **SETTINGS['rules']),
            'content': cf.content(m, prog, sorted(snaps[name].pool)), 'habit': features.habit_scores(snaps[name], views)}


tables = {}
for name, s in snaps.items():
    t = time.time()
    users = test_users if name == 'S31' else s.target.index
    tables[name] = features.build(s, views, users, cf_models(name), fallbacks[name], prog, n_cf=N_CF, n_pop=N_POP)
    gc.collect()
    print(f'{name}: history weeks {snapshot.FIRST_WEEK} to {s.cutoff_week - 1}, target weeks {s.cutoff_week} to {s.end_week}, '
          f'{len(tables[name]):,} rows for {tables[name].user_id.nunique()} users, {time.time() - t:.0f}s')

Windows-11-10.0.26200-SP0 | python 3.12.14 | numpy 2.5.4 | pandas 3.0.6 | lightgbm 4.7.0


collaborative settings chosen in cf_models: {'als': {'factors': 32, 'alpha': 1.0, 'reg': 30.0}, 'ppr': {'alpha': 0.5, 'weighting': 'count'}, 'rules': {'min_support': 30}, 'content': {}}


S14: history weeks 11 to 13, target weeks 14 to 20, 623,640 rows for 4080 users, 7s


S16: history weeks 11 to 15, target weeks 16 to 20, 630,724 rows for 3787 users, 9s


S18: history weeks 11 to 17, target weeks 18 to 20, 571,405 rows for 3232 users, 10s


S21: history weeks 11 to 20, target weeks 21 to 30, 956,649 rows for 4030 users, 18s


S26: history weeks 11 to 25, target weeks 26 to 30, 943,123 rows for 3664 users, 24s


S31: history weeks 11 to 30, target weeks 31 to 42, 471,705 rows for 1260 users, 29s


In [2]:
# Candidate recall per snapshot
rows = {}
for name in ['S14', 'S16', 'S18', 'S21', 'S26']:
    df, s = tables[name], snaps[name]
    m = s.target.explode().astype('int64').reset_index().merge(df, on=['user_id', 'tv_show_id'], how='left')
    rows[name] = {'candidates per user, median': df.groupby('user_id').size().median(), 'recall': m.src_pop.notna().mean(),
                  'positives': int(df.y.sum())}
display(pd.DataFrame(rows).round(4))

,S14,S16,S18,S21,S26
"candidates per user, median",150.0000,161.0000,171.0000,223.0000,241.0000
recall,0.8083,0.7851,0.7567,0.7861,0.7664
positives,16489.0000,14865.0000,12228.0000,15840.0000,14041.0000


In [3]:
# Experiments: train on labels before week 21, evaluate on S21
def map_s21(score):
    df = tables['S21']
    sc = df[['user_id', 'tv_show_id']].assign(score=ranker.user_rank_score(df, score))
    r = recs.top5(sc, snaps['S21'].target.index, fallbacks['S21'], snaps['S21'].pool)
    return recs.evaluate(r, snaps['S21'].target, {'test users': test_users}), r


EXPERIMENTS = {'E1': (['S16'], 'base'), 'E2': (['S14', 'S16', 'S18'], 'base'), 'E3': (['S14', 'S16', 'S18'], 'regularised')}
OBJS = ['binary', 'lambdarank']
results, preds, iters = {}, {}, {}
for exp, (train_names, pset) in EXPERIMENTS.items():
    train_df = pd.concat([tables[n] for n in train_names], ignore_index=True)
    for obj in OBJS:
        t = time.time()
        b, it = ranker.fit(train_df, obj, params=ranker.PARAM_SETS[pset])
        preds[(exp, obj)] = ranker.predict(b, tables['S21'])
        iters[(exp, obj)] = it
        res, _ = map_s21(preds[(exp, obj)])
        results[f'{exp} {obj}'] = {**res, 'train rows': len(train_df), 'best iteration': it, 'seconds': round(time.time() - t)}
        print(exp, obj, results[f'{exp} {obj}'], flush=True)
    rr = sum(ranker.user_rank_score(tables['S21'], preds[(exp, o)]) for o in OBJS)
    res, _ = map_s21(rr)
    results[f'{exp} binary + lambdarank'] = res
    del train_df
    gc.collect()
display(pd.DataFrame(results).T.round(4))

E1 binary {'all': 0.3536443341604632, 'test users': 0.39364637681159415, 'train rows': 630724, 'best iteration': 153, 'seconds': 7}


E1 lambdarank {'all': 0.3561290322580645, 'test users': 0.3956, 'train rows': 630724, 'best iteration': 66, 'seconds': 5}


E2 binary {'all': 0.3520678246484698, 'test users': 0.38827826086956524, 'train rows': 1825769, 'best iteration': 301, 'seconds': 20}


E2 lambdarank {'all': 0.35682133995037213, 'test users': 0.3948985507246377, 'train rows': 1825769, 'best iteration': 137, 'seconds': 17}


E3 binary {'all': 0.3554474772539289, 'test users': 0.39191014492753623, 'train rows': 1825769, 'best iteration': 918, 'seconds': 50}


E3 lambdarank {'all': 0.35721836228287845, 'test users': 0.39394202898550723, 'train rows': 1825769, 'best iteration': 361, 'seconds': 32}


,all,test users,train rows,best iteration,seconds
E1 binary,0.3536,0.3936,630724.0,153.0,7.0
E1 lambdarank,0.3561,0.3956,630724.0,66.0,5.0
E1 binary + lambdarank,0.3554,0.3945,NaN,NaN,NaN
E2 binary,0.3521,0.3883,1825769.0,301.0,20.0
E2 lambdarank,0.3568,0.3949,1825769.0,137.0,17.0
E2 binary + lambdarank,0.3554,0.3923,NaN,NaN,NaN
E3 binary,0.3554,0.3919,1825769.0,918.0,50.0
E3 lambdarank,0.3572,0.3939,1825769.0,361.0,32.0
E3 binary + lambdarank,0.3574,0.3935,NaN,NaN,NaN


In [4]:
# Selection on S21 (all target users) and the S21 lists of the selected model for the analysis notebook
best_name = max(results, key=lambda k: results[k]['all'])
best_exp = best_name.split()[0]
best_objs = OBJS if best_name.endswith('binary + lambdarank') else [best_name.split()[1]]
print('selected:', best_name, '| objectives', best_objs, '| S21 MAP@5', round(results[best_name]['all'], 4))
score = sum(ranker.user_rank_score(tables['S21'], preds[(best_exp, o)]) for o in best_objs)
_, lists_s21 = map_s21(score)
recs.write_submission(lists_s21, snaps['S21'].target.index, os.path.join(ROOT, 'modelling', 's21_ranker_v2_top5.csv'))

selected: E3 binary + lambdarank | objectives ['binary', 'lambdarank'] | S21 MAP@5 0.3574


In [5]:
# Final: refit the selected experiment on every snapshot whose labels end by week 30 and predict the test window
train_names, pset = EXPERIMENTS[best_exp]
final_names = train_names + ['S21', 'S26'] if len(train_names) > 1 else train_names + ['S21']
for n in [n for n in tables if n not in final_names + ['S31']]:
    del tables[n]
final_df = pd.concat([tables[n] for n in final_names], ignore_index=True)
for n in final_names:
    del tables[n]
gc.collect()
print('final training snapshots', final_names, '| rows', len(final_df))
t31 = tables['S31']
p31 = {}
for obj in best_objs:
    t = time.time()
    b = ranker.refit(final_df, obj, int(iters[(best_exp, obj)] * 1.2), params=ranker.PARAM_SETS[pset])
    p31[obj] = ranker.predict(b, t31)
    print(obj, 'refitted in', round(time.time() - t), 's')
score31 = sum(ranker.user_rank_score(t31, p31[o]) for o in best_objs)
sc = t31[['user_id', 'tv_show_id']].assign(score=ranker.user_rank_score(t31, score31))
final_lists = recs.top5(sc, test_users, fallbacks['S31'], snaps['S31'].pool)
recs.write_submission(final_lists, test_users, os.path.join(SUB_DIR, 'ranker_v2.csv'))
imp = pd.Series(b.feature_importance('gain'), index=b.feature_name()).sort_values(ascending=False)
display((imp / imp.sum()).head(25).round(4).to_frame(f'gain share, final {best_objs[-1]} model').T)
with open(os.path.join(ROOT, 'modelling', 'ranker_v2_results.json'), 'w') as f:
    json.dump({'results': results, 'selected': best_name, 'final_snapshots': final_names, 'param_set': pset,
               'iterations': {f'{k[0]} {k[1]}': v for k, v in iters.items()}, 'n_cf': N_CF, 'n_pop': N_POP,
               'importance': (imp / imp.sum()).round(5).to_dict()}, f, indent=1, default=str)
print(f'total runtime {time.time() - T0:.0f}s')

final training snapshots ['S14', 'S16', 'S18', 'S21', 'S26'] | rows 3725541


binary refitted in 124 s


lambdarank refitted in 91 s


,rank_habit_exposure,rank_cf_habit,rank_cf_ppr,habit_exposure,pop_top5,n_touch,cf_habit,rank_hs,rank_cf_als,win_airings,n_half,rank_cf_rules,win_airings_user_channels,u_q_total,rank_hs_p0,pop_q_users,u_q_5th,pop_q_last4,airing_ratio,category,cf_rules,hist_airings,weeks_since_last,hs_p0,pop_q_per_user
"gain share, final lambdarank model",0.403,0.1467,0.1186,0.0743,0.0293,0.0269,0.0232,0.0217,0.02,0.0193,0.014,0.0101,0.0094,0.0091,0.0065,0.0058,0.005,0.0045,0.0042,0.0039,0.0037,0.003,0.003,0.0029,0.0028


total runtime 463s
